# Full PRR training — reviewer reproduction

This notebook reconstructs the **24 full Predictive Relevance Retrieval (PRR) stacks**
used by the aligned long-horizon benchmark. It consolidates the frozen source training
pipelines into one reproducibility-focused implementation.

## Exact training protocol recovered from the original sources

- retrieval lookback: 96
- memory stride: 8
- Pearson candidate branch: Top-100
- learned-embedding branch: Top-100
- final Top-K: 10
- train memory/query split: first 60% of the training region is retrieval memory;
  the remainder supplies train-only supervision queries
- query-anchor stride during full-PRR training: 4
- encoder candidate list: 100 candidates/query = 50 Pearson-hard + 50 random
- query/history encoders: separate Conv1d networks, 64-D unit-normalized output
- encoder teacher temperature: 0.5
- encoder student temperature: 0.10
- encoder: 20 epochs, batch 32, AdamW lr 1e-3, wd 1e-4, grad clip 5
- reranker: 285-D input, hidden 384 x 2, dropout 0.1
- reranker loss: listwise future-compatible target + Top-K-aware pairwise softplus,
  pairwise weight 1.0
- reranker: batch 64, AdamW lr 1e-3, wd 1e-4, max 30 epochs,
  patience 6, grad clip 5
- internal reranker Phase-A split: chronological 80/20 split of train-only
  supervision anchors
- final reranker: fresh initialization and refit on all train-only records for the
  validation-selected number of epochs
- residual scale is positive, trainable, and initialized at 0.1
- random seed: 0

## Compute-matched multi-dataset training budget

ETTh1 uses all 7 channels and all eligible train-only query anchors.
The other five datasets use at most 64 evenly spaced training channels and at most
6,000 channel-anchor training pairs. **Evaluation still uses all channels.**

## ETTh1 H=192 provenance

The original ETTh1 pipeline reused a previously validated H=192 stack when that
checkpoint was present. For a standalone reproduction, this notebook can retrain H=192
from the same frozen recipe. Set `ETTH1_H192_SOURCE` if the original prior checkpoint
is available and exact lineage reuse is desired.

By default, the notebook performs only configuration/data audits. Set
`RUN_TRAINING = True` in the final section to train checkpoints.


In [ ]:
# Cell 1 — imports and package root

from pathlib import Path
from dataclasses import dataclass, asdict
import os, gc, json, math, random, shutil, warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 320)
pd.set_option("display.width", 680)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_package_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "experiments" / "prr_long_horizon").exists():
            return p
    env = os.environ.get("PRR_ROOT")
    if env:
        return Path(env).resolve()
    # Running from an arbitrary location is allowed; outputs remain local.
    return start

ROOT = find_package_root()
OUT_ROOT = ROOT / "artifacts" / "full_prr_training"
STACK_ROOT = OUT_ROOT / "full_stacks"
LOG_ROOT = OUT_ROOT / "logs"
for p in [OUT_ROOT, STACK_ROOT, LOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("ROOT   :", ROOT)
print("OUTPUT :", OUT_ROOT)
print("torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
print("DEVICE :", DEVICE)

In [ ]:
# Cell 2 — recovered frozen training configuration

@dataclass
class CFG:
    datasets: tuple = (
        "ETTh1", "Weather", "Electricity",
        "Traffic", "Exchange", "Solar",
    )
    horizons: tuple = (96, 192, 336, 720)

    retrieval_seq_len: int = 96
    memory_stride: int = 8
    top_k: int = 10
    top_m_pattern: int = 100
    top_m_learned: int = 100
    union_max: int = 200

    train_memory_fraction: float = 0.60
    train_query_stride: int = 4
    phase_a_train_fraction: float = 0.80

    # Original Exp12 compute-matched screen budget.
    screen_max_train_channels: int = 64
    screen_max_train_pairs: int = 6000

    emb_dim: int = 64
    enc_hidden: int = 128
    enc_candidates: int = 100
    enc_hard_fraction: float = 0.50
    enc_epochs: int = 20
    enc_batch: int = 32
    enc_lr: float = 1e-3
    enc_wd: float = 1e-4
    teacher_tau: float = 0.5
    student_tau: float = 0.10

    rerank_hidden: int = 384
    rerank_dropout: float = 0.1
    rerank_batch: int = 64
    rerank_lr: float = 1e-3
    rerank_wd: float = 1e-4
    rerank_max_epochs: int = 30
    rerank_patience: int = 6
    pairwise_lambda: float = 1.0
    grad_clip: float = 5.0

    feature_clip: float = 8.0
    iqr_floor: float = 1e-5
    seed: int = 0

cfg = CFG()
DATASETS = list(cfg.datasets)
HORIZONS = list(cfg.horizons)
EPS = 1e-8

print(json.dumps(asdict(cfg), indent=2))

In [ ]:
# Cell 3 — reproducibility

def seed_everything(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(cfg.seed)

In [ ]:
# Cell 4 — dataset discovery and train-only normalization

EXPECTED_CHANNELS = {
    "ETTh1": 7,
    "Weather": 21,
    "Electricity": 321,
    "Traffic": 862,
    "Exchange": 8,
    "Solar": 137,
}

DATA_CANDIDATES = {
    "ETTh1": [
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
        Path("/data/pcw_workspace/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/code/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/dataset/ETT-small/ETTh1.csv"),
        Path("/data/dataset/ETTh1.csv"),
    ],
    "Weather": [
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
        Path("/code/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path("/data/dataset/weather.csv"),
    ],
    "Electricity": [
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
        Path("/code/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity.csv"),
    ],
    "Traffic": [
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
        Path("/code/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic.csv"),
    ],
    "Exchange": [
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/pcw_workspace/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/code/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate.csv"),
    ],
    "Solar": [
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
        Path("/code/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/solar_AL.txt"),
    ],
}

def resolve_data(name):
    hit = next((p for p in DATA_CANDIDATES[name] if p.is_file()), None)
    if hit is None:
        raise FileNotFoundError(
            f"{name}: dataset not found. See data/README.md and DATA_CANDIDATES."
        )
    return hit

def clean_numeric_frame(df):
    return (
        df.apply(pd.to_numeric, errors="coerce")
        .dropna(axis=1, how="all")
        .replace([np.inf, -np.inf], np.nan)
        .interpolate(axis=0, limit_direction="both")
        .ffill()
        .bfill()
    )

def load_dataset(name):
    path = resolve_data(name)

    if name == "Solar":
        df = pd.read_csv(path, header=None)
        if df.shape[1] == 1:
            df = pd.read_csv(path, header=None, sep=r"\s+")
        numeric = clean_numeric_frame(df)
    else:
        df = pd.read_csv(path)
        date_col = next(
            (c for c in df.columns
             if str(c).lower() in {"date","datetime","timestamp","time"}),
            None,
        )
        value_df = df.drop(columns=[date_col]) if date_col is not None else df.copy()

        # Preserve original Weather/Electricity iTransformer column ordering.
        if name in {"Weather", "Electricity"} and "OT" in value_df.columns:
            value_df = value_df[[c for c in value_df.columns if c != "OT"] + ["OT"]]

        numeric = clean_numeric_frame(value_df)

    raw = numeric.to_numpy(dtype=np.float32)
    if raw.shape[1] != EXPECTED_CHANNELS[name]:
        raise ValueError(
            f"{name}: expected {EXPECTED_CHANNELS[name]} channels, got {raw.shape[1]}"
        )

    n = len(raw)
    if name == "ETTh1":
        if raw.shape != (17420, 7):
            raise ValueError(f"ETTh1: expected (17420,7), got {raw.shape}")
        train_end, val_end, test_end = 8640, 11520, n
    else:
        train_end = int(0.70 * n)
        val_end = n - int(0.20 * n)
        test_end = n

    mean = raw[:train_end].mean(axis=0).astype(np.float32)
    std = raw[:train_end].std(axis=0, ddof=0).astype(np.float32)

    bad = np.flatnonzero(std <= 1e-6)
    if len(bad):
        raise ValueError(f"{name}: train-degenerate channels {bad[:40].tolist()}")

    z = ((raw - mean[None, :]) / std[None, :]).astype(np.float32)

    return {
        "name": name,
        "path": path,
        "raw": raw,
        "z": z,
        "n_time": n,
        "n_channels": raw.shape[1],
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
        "train_mean": mean,
        "train_std": std,
    }

DATA = {name: load_dataset(name) for name in DATASETS}

dataset_audit = pd.DataFrame([
    {
        "Dataset": d,
        "Rows": DATA[d]["n_time"],
        "Channels": DATA[d]["n_channels"],
        "TrainEnd": DATA[d]["train_end"],
        "ValEnd": DATA[d]["val_end"],
        "TestEnd": DATA[d]["test_end"],
        "Path": str(DATA[d]["path"]),
    }
    for d in DATASETS
])
display(dataset_audit)
dataset_audit.to_csv(LOG_ROOT / "dataset_audit.csv", index=False)

In [ ]:
# Cell 5 — retrieval windows and seven past-only statistics

def batch_pattern(x):
    x = np.asarray(x, dtype=np.float32)
    xc = x - x.mean(axis=-1, keepdims=True)
    n = np.linalg.norm(xc, axis=-1, keepdims=True)
    return np.where(
        n > EPS,
        xc / np.maximum(n, EPS),
        0.0,
    ).astype(np.float32)

def context7(x):
    x = np.asarray(x, dtype=np.float32)
    short = max(8, cfg.retrieval_seq_len // 4)

    m = x.mean(axis=-1)
    s = x.std(axis=-1) + EPS

    f1 = (x[..., -1] - m) / s
    f2 = (x[..., -short:].mean(axis=-1) - m) / s
    f3 = (x[..., -1] - x[..., -short]) / s
    f4 = (x[..., -1] - x[..., 0]) / s

    dfull = np.diff(x, axis=-1)
    dshort = np.diff(x[..., -short:], axis=-1)
    f5 = (dshort.std(axis=-1) + EPS) / (dfull.std(axis=-1) + EPS)

    t = np.linspace(-1.0, 1.0, cfg.retrieval_seq_len, dtype=np.float32)
    t = t - t.mean()
    f6 = (
        np.sum(t * (x - m[..., None]), axis=-1)
        / (np.sum(t * t) + EPS)
    ) / s

    a = x[..., :-1]
    b = x[..., 1:]
    a = a - a.mean(axis=-1, keepdims=True)
    b = b - b.mean(axis=-1, keepdims=True)
    f7 = (
        np.sum(a * b, axis=-1)
        / (
            np.sqrt(
                np.sum(a * a, axis=-1)
                * np.sum(b * b, axis=-1)
            )
            + EPS
        )
    )

    return np.stack(
        [f1, f2, f3, f4, f5, f6, f7],
        axis=-1,
    ).astype(np.float32)

def extract_channel(z, channel, anchors, horizon):
    anchors = np.asarray(anchors, dtype=np.int64)

    pi = (
        anchors[:, None]
        - cfg.retrieval_seq_len
        + np.arange(cfg.retrieval_seq_len)[None, :]
    )
    fi = anchors[:, None] + np.arange(int(horizon))[None, :]

    past = z[pi, channel].astype(np.float32)
    future = z[fi, channel].astype(np.float32)
    current = z[anchors - 1, channel].astype(np.float32)

    return past, (future - current[:, None]).astype(np.float32)

In [ ]:
# Cell 6 — full PRR architecture

class TemporalEncoder1D(nn.Module):
    def __init__(self, hidden=128, emb_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, hidden // 2, 5, stride=2, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden // 2, hidden, 5, stride=2, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden, hidden, 3, stride=2, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj = nn.Linear(hidden, emb_dim)

    def forward(self, x):
        if x.ndim == 2:
            x = x[:, None, :]
        h = self.net(x).squeeze(-1)
        return F.normalize(self.proj(h), dim=-1)

def inverse_softplus(x):
    return math.log(math.exp(float(x)) - 1.0)

class WideFusionReranker(nn.Module):
    def __init__(self):
        super().__init__()
        in_dim = 1 + 4 * (7 + cfg.emb_dim)  # 285
        h = cfg.rerank_hidden
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h, h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(cfg.rerank_dropout),
            nn.Linear(h, 1),
        )
        self.raw_alpha = nn.Parameter(
            torch.tensor(inverse_softplus(0.1), dtype=torch.float32)
        )

    def forward(self, feat, base, mask):
        delta = self.net(feat).squeeze(-1)
        alpha = F.softplus(self.raw_alpha)
        score = base + alpha * delta
        return score.masked_fill(~mask, -1e9)

def make_encoder_pair(seed):
    seed_everything(seed)
    qenc = TemporalEncoder1D(cfg.enc_hidden, cfg.emb_dim).to(DEVICE)
    henc = TemporalEncoder1D(cfg.enc_hidden, cfg.emb_dim).to(DEVICE)
    return qenc, henc

def make_reranker(seed):
    seed_everything(seed)
    return WideFusionReranker().to(DEVICE)

@torch.no_grad()
def encode_rows(enc, x, batch=512):
    out = []
    enc.eval()
    for left in range(0, len(x), batch):
        t = torch.from_numpy(x[left:left+batch]).float().to(DEVICE)
        out.append(enc(t).cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float32)

In [ ]:
# Cell 7 — checkpoint serialization

class FullPRRStack:
    def __init__(
        self, qenc, henc, reranker, stats_med, stats_iqr,
        dataset, horizon, training_channels, training_query_anchors,
        rerank_best_epoch,
    ):
        self.qenc = qenc.eval()
        self.henc = henc.eval()
        self.reranker = reranker.eval()
        self.stats_med = np.asarray(stats_med, np.float32)
        self.stats_iqr = np.asarray(stats_iqr, np.float32)
        self.dataset = str(dataset)
        self.horizon = int(horizon)
        self.training_channels = np.asarray(training_channels, np.int64)
        self.training_query_anchors = np.asarray(training_query_anchors, np.int64)
        self.rerank_best_epoch = int(rerank_best_epoch)

def reviewer_stack_path(dataset, horizon):
    suffix = "Full" if dataset == "ETTh1" else "screen"
    return STACK_ROOT / f"{dataset}_H{int(horizon)}_{suffix}.pt"

def save_stack(stack, path):
    torch.save(
        {
            "query_encoder": stack.qenc.state_dict(),
            "history_encoder": stack.henc.state_dict(),
            "reranker": stack.reranker.state_dict(),
            "stats_med": stack.stats_med,
            "stats_iqr": stack.stats_iqr,
            "dataset": stack.dataset,
            "horizon": stack.horizon,
            "training_channels": stack.training_channels,
            "training_query_anchors": stack.training_query_anchors,
            "rerank_best_epoch": stack.rerank_best_epoch,
            "cfg": asdict(cfg),
        },
        path,
    )

In [ ]:
# Cell 8 — exact dataset-specific train-only training geometry

def evenly_spaced_indices(n, k):
    n = int(n)
    k = min(int(k), n)
    if k >= n:
        return np.arange(n, dtype=np.int64)

    idx = np.linspace(0, n - 1, num=k)
    idx = np.unique(np.round(idx).astype(np.int64))

    if len(idx) < k:
        unused = np.setdiff1d(np.arange(n, dtype=np.int64), idx)
        idx = np.sort(np.concatenate([idx, unused[:k-len(idx)]]))
    return idx[:k]

def training_geometry(data, horizon):
    horizon = int(horizon)
    train_end = int(data["train_end"])

    memory_end = int(cfg.train_memory_fraction * train_end)

    # Preserve the original ETTh1 training safety bound.
    if data["name"] == "ETTh1":
        memory_end = max(
            memory_end,
            cfg.retrieval_seq_len + horizon + cfg.memory_stride,
        )

    memory_anchors = np.arange(
        cfg.retrieval_seq_len,
        memory_end - horizon + 1,
        cfg.memory_stride,
        dtype=np.int64,
    )

    all_query_anchors = np.arange(
        memory_end,
        train_end - horizon + 1,
        cfg.train_query_stride,
        dtype=np.int64,
    )

    if data["name"] == "ETTh1":
        training_channels = np.arange(data["n_channels"], dtype=np.int64)
        query_anchors = all_query_anchors
    else:
        n_channels = min(
            data["n_channels"],
            cfg.screen_max_train_channels,
        )
        training_channels = evenly_spaced_indices(
            data["n_channels"],
            n_channels,
        )

        max_q = max(
            4,
            cfg.screen_max_train_pairs // len(training_channels),
        )
        qpos = evenly_spaced_indices(
            len(all_query_anchors),
            min(len(all_query_anchors), max_q),
        )
        query_anchors = all_query_anchors[qpos]

    return {
        "memory_end": memory_end,
        "memory_anchors": memory_anchors,
        "all_query_anchors": all_query_anchors,
        "training_channels": training_channels,
        "query_anchors": query_anchors,
        "training_pairs": len(training_channels) * len(query_anchors),
    }

geometry_rows = []
for d in DATASETS:
    for h in HORIZONS:
        g = training_geometry(DATA[d], h)
        geometry_rows.append({
            "Dataset": d,
            "H": h,
            "MemoryEnd": g["memory_end"],
            "MemoryPerChannel": len(g["memory_anchors"]),
            "TrainingChannels": len(g["training_channels"]),
            "TrainingQueryAnchors": len(g["query_anchors"]),
            "TrainingPairs": g["training_pairs"],
        })

geometry_audit = pd.DataFrame(geometry_rows)
display(geometry_audit)
geometry_audit.to_csv(LOG_ROOT / "training_geometry_audit.csv", index=False)

In [ ]:
# Cell 9 — build train-only retrieval problem

def build_training_problem(data, horizon):
    horizon = int(horizon)
    z = data["z"]
    geom = training_geometry(data, horizon)

    memory_anchors = geom["memory_anchors"]
    channels = geom["training_channels"]

    M = len(memory_anchors)
    C = len(channels)

    if M < cfg.top_k:
        raise ValueError(f"{data['name']} H={horizon}: too few memory candidates")
    if len(geom["query_anchors"]) < 4:
        raise ValueError(f"{data['name']} H={horizon}: too few training queries")

    mem_past = np.empty((C, M, cfg.retrieval_seq_len), dtype=np.float32)
    mem_future = np.empty((C, M, horizon), dtype=np.float32)
    mem_pattern = np.empty_like(mem_past)
    mem_stats_raw = np.empty((C, M, 7), dtype=np.float32)

    for slot, c in enumerate(channels):
        p, f = extract_channel(z, int(c), memory_anchors, horizon)
        mem_past[slot] = p
        mem_future[slot] = f
        mem_pattern[slot] = batch_pattern(p)
        mem_stats_raw[slot] = context7(p)

    flat_stats = mem_stats_raw.reshape(-1, 7)
    stats_med = np.median(flat_stats, axis=0).astype(np.float32)
    q25 = np.percentile(flat_stats, 25, axis=0)
    q75 = np.percentile(flat_stats, 75, axis=0)
    stats_iqr = (q75 - q25).astype(np.float32)
    stats_iqr = np.where(
        stats_iqr < cfg.iqr_floor,
        1.0,
        stats_iqr,
    ).astype(np.float32)

    mem_stats = np.clip(
        (mem_stats_raw - stats_med) / stats_iqr,
        -cfg.feature_clip,
        cfg.feature_clip,
    ).astype(np.float32)

    return {
        **geom,
        "dataset": data["name"],
        "horizon": horizon,
        "M": M,
        "mem_past": mem_past,
        "mem_future": mem_future,
        "mem_pattern": mem_pattern,
        "mem_stats": mem_stats,
        "stats_med": stats_med,
        "stats_iqr": stats_iqr,
    }

In [ ]:
# Cell 10 — future-supervised dual-encoder training

class EncoderRecordDataset(Dataset):
    def __init__(self, records, data, problem):
        self.records = records
        self.data = data
        self.problem = problem

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        qpast, _ = extract_channel(
            self.data["z"],
            r["channel"],
            np.array([r["anchor"]], dtype=np.int64),
            self.problem["horizon"],
        )
        cpast = self.problem["mem_past"][
            r["channel_slot"], r["cand_idx"]
        ]
        return (
            torch.from_numpy(qpast[0]),
            torch.from_numpy(cpast),
            torch.from_numpy(r["dist"]),
        )

def build_encoder_records(data, problem, seed):
    # Preserve the original source-specific deterministic seed formulas.
    if data["name"] == "ETTh1":
        rng_seed = 11000 + seed + problem["horizon"] + data["train_end"]
    else:
        rng_seed = 12000 + seed + problem["horizon"]

    rng = np.random.default_rng(rng_seed)
    records = []
    qanchors = problem["query_anchors"]
    M = problem["M"]

    for slot, c in enumerate(problem["training_channels"]):
        qpast, qfuture = extract_channel(
            data["z"],
            int(c),
            qanchors,
            problem["horizon"],
        )
        qpat = batch_pattern(qpast)
        ps = qpat @ problem["mem_pattern"][slot].T

        n_hard = int(cfg.enc_candidates * cfg.enc_hard_fraction)
        n_rand = cfg.enc_candidates - n_hard

        for j, anchor in enumerate(qanchors):
            hard = np.argsort(-ps[j])[:n_hard]
            remain = np.setdiff1d(
                np.arange(M, dtype=np.int64),
                hard,
                assume_unique=False,
            )
            rand = rng.choice(
                remain,
                size=n_rand,
                replace=(len(remain) < n_rand),
            )
            cand_idx = np.concatenate([hard, rand]).astype(np.int64)
            rng.shuffle(cand_idx)

            dist = np.mean(
                (
                    problem["mem_future"][slot, cand_idx]
                    - qfuture[j][None, :]
                ) ** 2,
                axis=1,
            ).astype(np.float32)

            records.append({
                "anchor": int(anchor),
                "channel": int(c),
                "channel_slot": int(slot),
                "cand_idx": cand_idx,
                "dist": dist,
            })

    records.sort(key=lambda r: (r["anchor"], r["channel"]))
    return records

def train_dual_encoder(data, problem, seed):
    records = build_encoder_records(data, problem, seed)

    ds = EncoderRecordDataset(records, data, problem)
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        ds,
        batch_size=cfg.enc_batch,
        shuffle=True,
        generator=g,
        num_workers=0,
    )

    qenc, henc = make_encoder_pair(seed)
    params = list(qenc.parameters()) + list(henc.parameters())
    opt = torch.optim.AdamW(
        params,
        lr=cfg.enc_lr,
        weight_decay=cfg.enc_wd,
    )

    history = []

    for epoch in range(1, cfg.enc_epochs + 1):
        qenc.train()
        henc.train()
        total = 0.0
        count = 0

        for qx, cx, dist in loader:
            qx = qx.float().to(DEVICE)
            cx = cx.float().to(DEVICE)
            dist = dist.float().to(DEVICE)

            B, K, L = cx.shape

            qemb = qenc(qx)
            cemb = henc(
                cx.reshape(B * K, L)
            ).reshape(B, K, -1)

            sim = torch.einsum("bd,bkd->bk", qemb, cemb)

            # Original code standardizes realized future distances per query.
            mu = dist.mean(dim=1, keepdim=True)
            sd = dist.std(
                dim=1,
                keepdim=True,
                unbiased=False,
            ).clamp_min(1e-6)

            target = F.softmax(
                -((dist - mu) / sd) / cfg.teacher_tau,
                dim=1,
            )
            logp = F.log_softmax(
                sim / cfg.student_tau,
                dim=1,
            )

            # Cross entropy equals KL(target || student) up to H(target).
            loss = -(target * logp).sum(dim=1).mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, cfg.grad_clip)
            opt.step()

            total += float(loss.item()) * B
            count += B

        mean_loss = total / max(count, 1)
        history.append({"Epoch": epoch, "Loss": mean_loss})

        print(
            f"[Encoder {data['name']} H={problem['horizon']}] "
            f"ep={epoch:02d}/{cfg.enc_epochs} loss={mean_loss:.6f}"
        )

    qenc.eval()
    henc.eval()
    return qenc, henc, pd.DataFrame(history)

In [ ]:
# Cell 11 — candidate union and fusion-reranker records

def topk_np(scores, k):
    k = min(int(k), len(scores))
    part = np.argpartition(-scores, k - 1)[:k]
    return part[np.argsort(-scores[part])]

def pair_block(q, cand):
    qrep = np.repeat(q[None, :], len(cand), axis=0)
    return np.concatenate(
        [qrep, cand, qrep - cand, np.abs(qrep - cand)],
        axis=1,
    ).astype(np.float32)

def build_rerank_records(data, problem, qenc, henc):
    qanchors = problem["query_anchors"]
    C = len(problem["training_channels"])

    mem_emb = np.empty(
        (C, problem["M"], cfg.emb_dim),
        dtype=np.float32,
    )

    for slot in range(C):
        mem_emb[slot] = encode_rows(
            henc,
            problem["mem_past"][slot],
        )

    records = []

    for slot, c in enumerate(problem["training_channels"]):
        qpast, qfuture = extract_channel(
            data["z"],
            int(c),
            qanchors,
            problem["horizon"],
        )

        qpat = batch_pattern(qpast)
        qemb_all = encode_rows(qenc, qpast)

        ps_all = qpat @ problem["mem_pattern"][slot].T
        ls_all = qemb_all @ mem_emb[slot].T

        for j, anchor in enumerate(qanchors):
            ptop = topk_np(ps_all[j], cfg.top_m_pattern)
            ltop = topk_np(ls_all[j], cfg.top_m_learned)

            union = np.array(
                sorted(set(ptop.tolist()) | set(ltop.tolist())),
                dtype=np.int64,
            )

            # Union of two Top-100 lists is <= 200; retained for exact source logic.
            if len(union) > cfg.union_max:
                op = np.argsort(-ps_all[j])
                ol = np.argsort(-ls_all[j])
                rp = np.empty_like(op)
                rl = np.empty_like(ol)
                rp[op] = np.arange(1, len(op) + 1)
                rl[ol] = np.arange(1, len(ol) + 1)
                fused = 1.0 / (60 + rp) + 1.0 / (60 + rl)
                union = union[
                    np.argsort(-fused[union])[:cfg.union_max]
                ]

            dist = np.mean(
                (
                    problem["mem_future"][slot, union]
                    - qfuture[j][None, :]
                ) ** 2,
                axis=1,
            ).astype(np.float32)

            qstat = np.clip(
                (
                    context7(qpast[j:j+1])[0]
                    - problem["stats_med"]
                )
                / problem["stats_iqr"],
                -cfg.feature_clip,
                cfg.feature_clip,
            ).astype(np.float32)

            records.append({
                "anchor": int(anchor),
                "channel": int(c),
                "channel_slot": int(slot),
                "cand_idx": union,
                "base_score": ps_all[j, union].astype(np.float32),
                "future_dist": dist,
                "qstat": qstat,
                "qemb": qemb_all[j].astype(np.float32),
            })

    records.sort(key=lambda r: (r["anchor"], r["channel"]))

    unique_a = np.unique([r["anchor"] for r in records])
    cut_n = max(
        1,
        min(
            len(unique_a) - 1,
            int(cfg.phase_a_train_fraction * len(unique_a)),
        ),
    )
    cut_anchor = unique_a[cut_n]

    train_records = [r for r in records if r["anchor"] < cut_anchor]
    valid_records = [r for r in records if r["anchor"] >= cut_anchor]

    return records, train_records, valid_records, mem_emb

In [ ]:
# Cell 12 — hybrid listwise + Top-K-aware pairwise reranker objective

class RerankDataset(Dataset):
    def __init__(self, records, problem, mem_emb):
        self.records = records
        self.problem = problem
        self.mem_emb = mem_emb

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        slot = r["channel_slot"]
        ci = r["cand_idx"]

        qjoint = np.concatenate(
            [r["qstat"], r["qemb"]],
            axis=0,
        )
        cjoint = np.concatenate(
            [
                self.problem["mem_stats"][slot, ci],
                self.mem_emb[slot, ci],
            ],
            axis=1,
        )

        feat = np.concatenate(
            [
                r["base_score"][:, None],
                pair_block(qjoint, cjoint),
            ],
            axis=1,
        ).astype(np.float32)

        n, D = feat.shape
        feat_pad = np.zeros((cfg.union_max, D), dtype=np.float32)
        base_pad = np.zeros(cfg.union_max, dtype=np.float32)
        dist_pad = np.zeros(cfg.union_max, dtype=np.float32)
        mask = np.zeros(cfg.union_max, dtype=bool)

        feat_pad[:n] = feat
        base_pad[:n] = r["base_score"]
        dist_pad[:n] = r["future_dist"]
        mask[:n] = True

        return (
            torch.from_numpy(feat_pad),
            torch.from_numpy(base_pad),
            torch.from_numpy(dist_pad),
            torch.from_numpy(mask),
        )

def teacher_from_distance(dist, mask):
    m = mask.float()
    n = m.sum(dim=1, keepdim=True).clamp_min(1.0)
    mu = (dist * m).sum(dim=1, keepdim=True) / n
    var = (((dist - mu) ** 2) * m).sum(dim=1, keepdim=True) / n
    sd = torch.sqrt(var + 1e-6)

    logits = -((dist - mu) / sd) / cfg.teacher_tau
    logits = logits.masked_fill(~mask, -1e9)
    return F.softmax(logits, dim=1)

def hybrid_loss(score, dist, mask):
    target = teacher_from_distance(dist, mask)

    listwise = -(
        target * F.log_softmax(score, dim=1)
    ).sum(dim=1).mean()

    safe = dist.masked_fill(~mask, float("inf"))
    pos_idx = torch.topk(
        safe,
        k=cfg.top_k,
        dim=1,
        largest=False,
    ).indices

    top_mask = torch.zeros_like(mask)
    top_mask.scatter_(1, pos_idx, True)
    top_mask &= mask
    neg_mask = mask & (~top_mask)

    pos_score = torch.gather(score, 1, pos_idx)
    margin = pos_score.unsqueeze(2) - score.unsqueeze(1)

    pair_mask = neg_mask.unsqueeze(1).expand_as(margin)
    ploss = F.softplus(-margin) * pair_mask.float()

    denom = pair_mask.float().sum(dim=(1, 2)).clamp_min(1.0)
    pairwise = (ploss.sum(dim=(1, 2)) / denom).mean()

    return listwise + cfg.pairwise_lambda * pairwise

@torch.no_grad()
def rerank_val_afmse(model, loader):
    model.eval()
    vals = []

    for feat, base, dist, mask in loader:
        feat = feat.float().to(DEVICE)
        base = base.float().to(DEVICE)
        dist = dist.float().to(DEVICE)
        mask = mask.to(DEVICE)

        score = model(feat, base, mask)
        top = torch.topk(score, k=cfg.top_k, dim=1).indices
        selected = torch.gather(dist, 1, top)
        vals.append(selected.mean(dim=1).cpu().numpy())

    return float(np.concatenate(vals).mean())

In [ ]:
# Cell 13 — fit one full PRR stack

def fit_full_prr_stack(data, horizon, overwrite=False):
    horizon = int(horizon)
    path = reviewer_stack_path(data["name"], horizon)

    if path.is_file() and not overwrite:
        print("[SKIP existing]", path)
        return path

    print("\n" + "=" * 100)
    print(f"TRAIN FULL PRR | {data['name']} | H={horizon}")
    print("=" * 100)

    problem = build_training_problem(data, horizon)

    qenc, henc, enc_hist = train_dual_encoder(
        data,
        problem,
        cfg.seed,
    )
    enc_hist.to_csv(
        LOG_ROOT / f"{data['name']}_H{horizon}_encoder_history.csv",
        index=False,
    )

    all_records, train_records, valid_records, mem_emb = build_rerank_records(
        data,
        problem,
        qenc,
        henc,
    )

    train_ds = RerankDataset(train_records, problem, mem_emb)
    valid_ds = RerankDataset(valid_records, problem, mem_emb)

    g = torch.Generator().manual_seed(cfg.seed)
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.rerank_batch,
        shuffle=True,
        generator=g,
        num_workers=0,
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=cfg.rerank_batch,
        shuffle=False,
        num_workers=0,
    )

    reranker = make_reranker(cfg.seed)
    opt = torch.optim.AdamW(
        reranker.parameters(),
        lr=cfg.rerank_lr,
        weight_decay=cfg.rerank_wd,
    )

    best_val = float("inf")
    best_epoch = 1
    wait = 0
    history = []

    for epoch in range(1, cfg.rerank_max_epochs + 1):
        reranker.train()
        losses = []

        for feat, base, dist, mask in train_loader:
            feat = feat.float().to(DEVICE)
            base = base.float().to(DEVICE)
            dist = dist.float().to(DEVICE)
            mask = mask.to(DEVICE)

            score = reranker(feat, base, mask)
            loss = hybrid_loss(score, dist, mask)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                reranker.parameters(),
                cfg.grad_clip,
            )
            opt.step()
            losses.append(float(loss.item()))

        val_afmse = rerank_val_afmse(reranker, valid_loader)
        history.append({
            "Epoch": epoch,
            "TrainLoss": float(np.mean(losses)),
            "ValAnalogFutureMSE": val_afmse,
        })

        print(
            f"[Reranker {data['name']} H={horizon}] "
            f"ep={epoch:02d} valAFMSE={val_afmse:.6f}"
        )

        if val_afmse < best_val - 1e-8:
            best_val = val_afmse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1
            if wait >= cfg.rerank_patience:
                break

    pd.DataFrame(history).to_csv(
        LOG_ROOT / f"{data['name']}_H{horizon}_reranker_phaseA.csv",
        index=False,
    )

    # Exact source rule: fresh reranker, all train-only records,
    # validation-selected epoch count.
    all_ds = RerankDataset(all_records, problem, mem_emb)
    g2 = torch.Generator().manual_seed(cfg.seed)
    all_loader = DataLoader(
        all_ds,
        batch_size=cfg.rerank_batch,
        shuffle=True,
        generator=g2,
        num_workers=0,
    )

    reranker = make_reranker(cfg.seed)
    opt = torch.optim.AdamW(
        reranker.parameters(),
        lr=cfg.rerank_lr,
        weight_decay=cfg.rerank_wd,
    )

    for epoch in range(1, best_epoch + 1):
        reranker.train()

        for feat, base, dist, mask in all_loader:
            feat = feat.float().to(DEVICE)
            base = base.float().to(DEVICE)
            dist = dist.float().to(DEVICE)
            mask = mask.to(DEVICE)

            score = reranker(feat, base, mask)
            loss = hybrid_loss(score, dist, mask)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                reranker.parameters(),
                cfg.grad_clip,
            )
            opt.step()

        print(
            f"[Reranker refit {data['name']} H={horizon}] "
            f"{epoch:02d}/{best_epoch}"
        )

    stack = FullPRRStack(
        qenc=qenc,
        henc=henc,
        reranker=reranker,
        stats_med=problem["stats_med"],
        stats_iqr=problem["stats_iqr"],
        dataset=data["name"],
        horizon=horizon,
        training_channels=problem["training_channels"],
        training_query_anchors=problem["query_anchors"],
        rerank_best_epoch=best_epoch,
    )
    save_stack(stack, path)

    pd.DataFrame([{
        "Dataset": data["name"],
        "Horizon": horizon,
        "TrainingChannels": len(problem["training_channels"]),
        "FullChannels": data["n_channels"],
        "TrainingQueryAnchors": len(problem["query_anchors"]),
        "TrainingPairs": problem["training_pairs"],
        "MemoryPerChannel": problem["M"],
        "RerankerBestEpoch": best_epoch,
        "Checkpoint": str(path),
    }]).to_csv(
        LOG_ROOT / f"{data['name']}_H{horizon}_training_budget.csv",
        index=False,
    )

    del (
        problem, all_records, train_records, valid_records,
        mem_emb, train_ds, valid_ds, all_ds,
        train_loader, valid_loader, all_loader,
    )
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("[SAVED]", path)
    return path

In [ ]:
# Cell 14 — optional exact ETTh1 H=192 lineage reuse

# The original original ETTh1 driver reused a previously validated the earlier validated
# H=192 checkpoint if present. A reviewer normally leaves this unset and retrains
# with the standalone recipe above.
ETTH1_H192_SOURCE = os.environ.get("ETTH1_H192_SOURCE", "")

def maybe_reuse_etth1_h192():
    if not ETTH1_H192_SOURCE:
        return False

    src = Path(ETTH1_H192_SOURCE)
    if not src.is_file():
        raise FileNotFoundError(src)

    dst = reviewer_stack_path("ETTh1", 192)
    shutil.copy2(src, dst)
    print("[REUSED original ETTh1 H=192 lineage]", src, "->", dst)
    return True

In [ ]:
# Cell 15 — train selected or all 24 stacks

# Safe default: do not start a long GPU job merely by opening/running the notebook.
RUN_TRAINING = False
OVERWRITE = False

# Change this list for a targeted reproduction, e.g. [("Weather", 96)].
TARGETS = [(d, h) for d in DATASETS for h in HORIZONS]

if RUN_TRAINING:
    if ("ETTh1", 192) in TARGETS:
        maybe_reuse_etth1_h192()

    produced = []

    for dataset, horizon in TARGETS:
        if (
            dataset == "ETTh1"
            and horizon == 192
            and reviewer_stack_path(dataset, horizon).is_file()
            and ETTH1_H192_SOURCE
        ):
            produced.append(reviewer_stack_path(dataset, horizon))
            continue

        produced.append(
            fit_full_prr_stack(
                DATA[dataset],
                horizon,
                overwrite=OVERWRITE,
            )
        )

    print("\nProduced checkpoints:", len(produced))
else:
    print(
        "Training is disabled. Set RUN_TRAINING=True to reproduce the selected stacks."
    )

In [ ]:
# Cell 16 — checkpoint inventory / handoff to inference notebook

rows = []
for d in DATASETS:
    for h in HORIZONS:
        p = reviewer_stack_path(d, h)
        rows.append({
            "dataset": d,
            "horizon": h,
            "exists": p.is_file(),
            "size_MB": p.stat().st_size / 2**20 if p.is_file() else np.nan,
            "path": str(p),
        })

checkpoint_inventory = pd.DataFrame(rows)
display(checkpoint_inventory)
checkpoint_inventory.to_csv(
    OUT_ROOT / "checkpoint_inventory.csv",
    index=False,
)

n_complete = int(checkpoint_inventory["exists"].sum())
print(f"Complete stacks: {n_complete}/24")
print(
    "The reviewer inference notebook first checks this reviewer-training output "
    "and then falls back to the original experiment paths."
)